In [2]:
import numpy as np

# =========================================================
# INPUT PARAMETERS
# =========================================================

params = {
    # --- Black liquor ---
    "m_bl": 1.0,              # kg/s (dry solids)
    "cp_bl": 2.95,            # kJ/kg.K
    "T_bl": 400,              # K
    "HHV_bl": 14000,          # kJ/kg
    "O2_required": 1.5,       # kg O2/kg BL (approx)
    "O2_in_fuel": 0.1,        # kg O2/kg BL

    # --- Ambient ---
    "T_amb": 298,             # K

    # --- Air ---
    "excess_air_frac": 0.2,   # 20% excess air
    "humidity_air": 0.01,     # kg/kg
    "cp_air": 1.01,           # kJ/kg.K
    "T_air": 350,             # K

    # --- Flue gas ---
    "cp_fg": 1.02,            # kJ/kg.K
    "T_fg": 450,              # K

    # --- Sootblowing ---
    "m_sb": 0.05,             # kg/s
    "h_evap": 2257,           # kJ/kg
    "cp_vapor": 1.88,         # kJ/kg.K
    "T_sb": 450,              # K

    # --- Chemical losses ---
    "m_chem": 0.02,           # kg/s
    "h_chem": 5000,           # kJ/kg

    # --- Other losses ---
    "loss_smelt": 200,        # kW
    "loss_radi_frac": 0.02,
    "loss_unacc_frac": 0.01,
    "loss_margin_frac": 0.01,

    # --- Steam ---
    "h_steam": 3200,          # kJ/kg
    "h_fw": 500               # kJ/kg
}

# =========================================================
# AIR CALCULATIONS
# =========================================================

def theoretical_air(O2_required, O2_in_fuel, O2_fraction_air=0.232):
    return (O2_required - O2_in_fuel) / O2_fraction_air

def total_air(theoretical_air, excess_air_frac):
    return theoretical_air * (1 + excess_air_frac)

def air_with_humidity(dry_air, humidity):
    return dry_air * (1 + humidity)

def infiltration_air(theoretical_air, frac=0.03):
    return theoretical_air * frac


# =========================================================
# HEAT CALCULATIONS
# =========================================================

def sensible_heat(m_dot, cp, T, T_ref):
    return m_dot * cp * (T - T_ref)

def combustion_heat(m_bl, HHV):
    return m_bl * HHV

def sootblowing_loss(m_sb, h_evap, cp_vapor, T_sb, T_ref):
    return m_sb * (h_evap + cp_vapor * (T_sb - T_ref))

def chemical_loss(m, h):
    return m * h


# =========================================================
# STEAM + PERFORMANCE
# =========================================================

def steam_generation(Q_input, Q_losses):
    return Q_input - Q_losses

def steam_efficiency(Q_to_steam, Q_input):
    return (Q_to_steam / Q_input) * 100

def steam_flow(Q_to_steam, h_steam, h_fw):
    return Q_to_steam / (h_steam - h_fw)


# =========================================================
# MAIN MODEL
# =========================================================

def run_model(params):

    # -----------------------------
    # AIR SYSTEM
    # -----------------------------

    air_theoretical = theoretical_air(params["O2_required"], params["O2_in_fuel"])
    air_total = total_air(air_theoretical, params["excess_air_frac"])
    air_humid = air_with_humidity(air_total, params["humidity_air"])
    air_infiltration = infiltration_air(air_theoretical)

    m_air = air_humid + air_infiltration

    # assume flue gas ~ air + fuel
    m_fg = m_air + params["m_bl"]

    # -----------------------------
    # HEAT INPUTS
    # -----------------------------

    Q_comb = combustion_heat(params["m_bl"], params["HHV_bl"])
    Q_bl_sens = sensible_heat(params["m_bl"], params["cp_bl"],
                             params["T_bl"], params["T_amb"])
    Q_air = sensible_heat(m_air, params["cp_air"],
                          params["T_air"], params["T_amb"])

    Q_input = Q_comb + Q_bl_sens + Q_air

    # -----------------------------
    # LOSSES
    # -----------------------------

    loss_fg = sensible_heat(m_fg, params["cp_fg"],
                            params["T_fg"], params["T_amb"])

    loss_sb = sootblowing_loss(params["m_sb"], params["h_evap"],
                               params["cp_vapor"], params["T_sb"], params["T_amb"])

    loss_chem = chemical_loss(params["m_chem"], params["h_chem"])

    loss_radi = params["loss_radi_frac"] * Q_input
    loss_unacc = params["loss_unacc_frac"] * Q_input
    loss_margin = params["loss_margin_frac"] * Q_input

    Q_losses = (
        loss_fg +
        params["loss_smelt"] +
        loss_sb +
        loss_chem +
        loss_radi +
        loss_unacc +
        loss_margin
    )

    # -----------------------------
    # STEAM
    # -----------------------------

    Q_to_steam = steam_generation(Q_input, Q_losses)
    m_steam = steam_flow(Q_to_steam, params["h_steam"], params["h_fw"])
    efficiency = steam_efficiency(Q_to_steam, Q_input)

    # -----------------------------
    # OUTPUT
    # -----------------------------

    print("\n==============================")
    print(" RECOVERY BOILER RESULTS")
    print("==============================\n")

    print("---- AIR SYSTEM ----")
    print(f"Theoretical air:        {air_theoretical:10.3f} kg/s")
    print(f"Total air (excess):     {air_total:10.3f} kg/s")
    print(f"Air + humidity:         {air_humid:10.3f} kg/s")
    print(f"Infiltration air:       {air_infiltration:10.3f} kg/s")
    print(f"TOTAL air:              {m_air:10.3f} kg/s\n")

    print("---- HEAT INPUTS ----")
    print(f"Combustion heat:        {Q_comb:10.2f} kW")
    print(f"BL sensible heat:       {Q_bl_sens:10.2f} kW")
    print(f"Air sensible heat:      {Q_air:10.2f} kW")
    print(f"TOTAL heat input:       {Q_input:10.2f} kW\n")

    print("---- HEAT LOSSES ----")
    print(f"Flue gas loss:          {loss_fg:10.2f} kW")
    print(f"Sootblowing loss:       {loss_sb:10.2f} kW")
    print(f"Chemical loss:          {loss_chem:10.2f} kW")
    print(f"Radiation loss:         {loss_radi:10.2f} kW")
    print(f"Unaccounted loss:       {loss_unacc:10.2f} kW")
    print(f"Margin loss:            {loss_margin:10.2f} kW")
    print(f"Smelt loss:             {params['loss_smelt']:10.2f} kW")
    print(f"TOTAL losses:           {Q_losses:10.2f} kW\n")

    print("---- STEAM PRODUCTION ----")
    print(f"Heat to steam:          {Q_to_steam:10.2f} kW")
    print(f"Steam production:       {m_steam:10.4f} kg/s\n")

    print("---- PERFORMANCE ----")
    print(f"Boiler efficiency:      {efficiency:10.2f} %\n")

    return {
        "m_air": m_air,
        "Q_input": Q_input,
        "Q_losses": Q_losses,
        "Q_to_steam": Q_to_steam,
        "m_steam": m_steam,
        "efficiency": efficiency
    }


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":
    run_model(params)


 RECOVERY BOILER RESULTS

---- AIR SYSTEM ----
Theoretical air:             6.034 kg/s
Total air (excess):          7.241 kg/s
Air + humidity:              7.314 kg/s
Infiltration air:            0.181 kg/s
TOTAL air:                   7.495 kg/s

---- HEAT INPUTS ----
Combustion heat:          14000.00 kW
BL sensible heat:           300.90 kW
Air sensible heat:          393.63 kW
TOTAL heat input:         14694.53 kW

---- HEAT LOSSES ----
Flue gas loss:             1317.04 kW
Sootblowing loss:           127.14 kW
Chemical loss:              100.00 kW
Radiation loss:             293.89 kW
Unaccounted loss:           146.95 kW
Margin loss:                146.95 kW
Smelt loss:                 200.00 kW
TOTAL losses:              2331.96 kW

---- STEAM PRODUCTION ----
Heat to steam:            12362.57 kW
Steam production:           4.5787 kg/s

---- PERFORMANCE ----
Boiler efficiency:           84.13 %

